## Example 1: Run a flow's tasks on a Dask cluster

In [ ]:
from prefect import flow, task
from prefect_dask import DaskTaskRunner

# Example 1: submit tasks from a flow; DaskTaskRunner starts a small local Dask cluster for them
@task
def square(x: int) -> int:
    return x * x


@flow(task_runner=DaskTaskRunner(cluster_kwargs={"n_workers": 2, "threads_per_worker": 1, "processes": False}))
def sum_of_squares(n: int) -> int:
    futures = [square.submit(i) for i in range(n)]
    return sum(future.result() for future in futures)


result = sum_of_squares(10)
assert result == sum(i * i for i in range(10)), result
print(f"Example 1 OK: sum of squares 0..9 on Dask = {result}")

## Example 2: Map a task over inputs and chain the results

In [ ]:
from prefect import flow, task
from prefect_dask import DaskTaskRunner

# Example 2: task.map fans a task out over a list; the futures feed the next task directly
@task
def double(x: int) -> int:
    return 2 * x


@task
def total(values: list[int]) -> int:
    return sum(values)


@flow(task_runner=DaskTaskRunner(cluster_kwargs={"n_workers": 2, "processes": False}))
def pipeline(xs: list[int]) -> int:
    doubled = double.map(xs)
    return total.submit(doubled).result()


assert pipeline([1, 2, 3, 4]) == 20
print("Example 2 OK: mapped over 4 inputs and summed to 20")

## Example 3: Use the Dask client from inside a task

In [ ]:
import dask.array as da
from prefect import flow, task
from prefect_dask import DaskTaskRunner, get_dask_client

# Example 3: get_dask_client gives a task the cluster's client, so it can run a Dask collection there
@task
def mean_of_random_array() -> float:
    with get_dask_client() as client:
        array = da.ones((2_000, 2_000), chunks=(500, 500)) * 3.0
        return float(client.compute(array.mean()).result())


@flow(task_runner=DaskTaskRunner(cluster_kwargs={"n_workers": 1, "threads_per_worker": 2, "processes": False}))
def compute_on_cluster() -> float:
    return mean_of_random_array.submit().result()


value = compute_on_cluster()
assert value == 3.0, value
print(f"Example 3 OK: a 2000x2000 dask array computed on the flow's cluster, mean = {value}")

## Example 4: Connect to a cluster that is already running

In [ ]:
from distributed import LocalCluster
from prefect import flow, task
from prefect_dask import DaskTaskRunner

# Example 4: pass `address` to reuse an existing scheduler instead of starting a new cluster
@task
def shout(word: str) -> str:
    return word.upper() + "!"


with LocalCluster(n_workers=1, threads_per_worker=1, processes=False, dashboard_address=None) as cluster:
    @flow(task_runner=DaskTaskRunner(address=cluster.scheduler_address))
    def greet(words: list[str]) -> list[str]:
        return [future.result() for future in shout.map(words)]

    greeting = greet(["hello", "dask"])
    workers = len(cluster.scheduler_info["workers"])
assert greeting == ["HELLO!", "DASK!"], greeting
print(f"Example 4 OK: {greeting} computed on an existing cluster with {workers} worker")

## Example 5: Task failures surface as Prefect states

In [ ]:
from prefect import flow, task
from prefect_dask import DaskTaskRunner

# Example 5: a task that raises on a Dask worker ends Failed; the other tasks still complete
@task(retries=0)
def checked_inverse(x: int) -> float:
    if x == 0:
        raise ZeroDivisionError("cannot invert 0")
    return 1 / x


@flow(task_runner=DaskTaskRunner(cluster_kwargs={"n_workers": 1, "processes": False}))
def invert_all(xs: list[int]) -> list[tuple[str, str]]:
    futures = checked_inverse.map(xs)
    for future in futures:
        future.wait()
    return [(future.state.type.value, str(future.state.message)) for future in futures]


outcomes = invert_all([1, 0, 4])
assert [kind for kind, _ in outcomes] == ["COMPLETED", "FAILED", "COMPLETED"], outcomes
assert "cannot invert 0" in outcomes[1][1] or "ZeroDivisionError" in outcomes[1][1], outcomes[1]
print(f"Example 5 OK: outcomes = {[kind for kind, _ in outcomes]}; failure: {outcomes[1][1][:80]}")